In [ ]:
# ==========================================
# CONSULTAS A LA API
# ==========================================
import requests
import pandas as pd

# 1. Definición de URLs y parámetros de rango de fechas
BASE_URL = "https://data-charts-api.hexlet.app"
START_DATE = "2023-03-01"
END_DATE = "2023-09-01"

params = {
    'begin': START_DATE,
    'end': END_DATE
}

# 2. Petición a la API para VISITAS
response_visits = requests.get(f"{BASE_URL}/visits", params=params)

if response_visits.status_code == 200:
    data_visits = response_visits.json()
    df_visits_api = pd.DataFrame(data_visits)
    # Convertir campo datetime a formato de fecha
    df_visits_api['datetime'] = pd.to_datetime(df_visits_api['datetime'])
    print(f"✅ Visitas obtenidas de la API correctamente. Total de registros: {len(df_visits_api)}")
else:
    print(f"❌ Error al consultar /visits: Código {response_visits.status_code}")

# 3. Petición a la API para REGISTROS
response_regs = requests.get(f"{BASE_URL}/registrations", params=params)

if response_regs.status_code == 200:
    data_regs = response_regs.json()
    df_regs_api = pd.DataFrame(data_regs)
    # Convertir campo datetime a formato de fecha
    df_regs_api['datetime'] = pd.to_datetime(df_regs_api['datetime'])
    print(f"✅ Registros obtenidos de la API correctamente. Total de registros: {len(df_regs_api)}")
else:
    print(f"❌ Error al consultar /registrations: Código {response_regs.status_code}")

# 4. Exploración rápida de los DataFrames obtenidos por API
print("\n--- Vista previa Visitas API ---")
display(df_visits_api.head())

print("\n--- Vista previa Registros API ---")
display(df_regs_api.head())

In [ ]:
# ==========================================
# CÁLCULO DE MÉTRICAS Y TASA DE CONVERSIÓN
# ==========================================
import pandas as pd

# ------------------------------------------
# 1. Limpieza y filtrado de Visitas
# ------------------------------------------
# A. Excluir bots (User-Agent contiene 'bot' sin importar mayúsculas/minúsculas)
df_visits_clean = df_visits_api[
    ~df_visits_api['user_agent'].str.contains('bot', case=False, na=False)
].copy()

# B. Quedarse con la última visita para cada visit_id
df_visits_clean = df_visits_clean.sort_values('datetime').groupby('visit_id').last().reset_index()

# C. Extraer solo la fecha (YYYY-MM-DD)
df_visits_clean['date_group'] = df_visits_clean['datetime'].dt.floor('D')

# D. Agrupar visitas por fecha y plataforma
df_visits_grouped = df_visits_clean.groupby(['date_group', 'platform']).size().reset_index(name='visits')

# ------------------------------------------
# 2. Procesamiento y agrupación de Registros
# ------------------------------------------
# A. Extraer solo la fecha (YYYY-MM-DD)
df_regs_api['date_group'] = df_regs_api['datetime'].dt.floor('D')

# B. Agrupar registros por fecha y plataforma
df_regs_grouped = df_regs_api.groupby(['date_group', 'platform']).size().reset_index(name='registrations')

# ------------------------------------------
# 3. Combinación (Merge) y cálculo de conversión
# ------------------------------------------
# A. Unir DataFrames por date_group y platform (outer para no perder fechas/plataformas sin registros)
df_conversion = pd.merge(df_visits_grouped, df_regs_grouped, on=['date_group', 'platform'], how='left')

# B. Rellenar posibles NaNs con 0 para registros
df_conversion['registrations'] = df_conversion['registrations'].fillna(0).astype(int)

# C. Calcular porcentaje de conversión: (registrations / visits) * 100
df_conversion['conversion'] = (df_conversion['registrations'] / df_conversion['visits']) * 100

# D. Ordenar por fecha de la más antigua a la más reciente
df_conversion = df_conversion.sort_values('date_group').reset_index(drop=True)

# ------------------------------------------
# 4. Guardar resultado en formato JSON
# ------------------------------------------
df_conversion.to_json("./conversion.json")

print("✅ Archivo conversion.json generado con éxito.")
print("\n--- Vista previa del DataFrame Final ---")
display(df_conversion.head(10))

In [ ]:
# ==========================================
# AGREGAR DATOS DE CAMPAÑAS PUBLICITARIAS
# ==========================================
import pandas as pd

# ------------------------------------------
# 1. Carga y preparación del CSV de Ads
# ------------------------------------------
# Cargar el archivo ads.csv
df_ads_raw = pd.read_csv('ads.csv')

# Convertir columna date a datetime y extraer solo la fecha (date_group)
df_ads_raw['date_group'] = pd.to_datetime(df_ads_raw['date']).dt.floor('D')

# Agrupar los datos de publicidad por fecha (agregando cost y tomando utm_campaign)
df_ads_grouped = df_ads_raw.groupby('date_group').agg({
    'cost': 'sum',
    'utm_campaign': 'first'  # o la campaña activa del día
}).reset_index()

# ------------------------------------------
# 2. Agrupar visitas y registros solo por fecha
# ------------------------------------------
# Agrupar el DataFrame de conversión anterior por fecha (sumando visitas y registros)
df_metrics_daily = df_conversion.groupby('date_group').agg({
    'visits': 'sum',
    'registrations': 'sum'
}).reset_index()

# ------------------------------------------
# 3. Combinación (Merge) de Métricas + Campañas
# ------------------------------------------
df_ads_combined = pd.merge(df_metrics_daily, df_ads_grouped, on='date_group', how='left')

# Rellenar valores nulos requeridos por el enunciado:
# cost -> 0 si no hubo inversión
df_ads_combined['cost'] = df_ads_combined['cost'].fillna(0)

# utm_campaign -> 'none' si no hubo campaña activa
df_ads_combined['utm_campaign'] = df_ads_combined['utm_campaign'].fillna('none')

# Reordenar las columnas al orden exacto especificado
df_ads_combined = df_ads_combined[['date_group', 'visits', 'registrations', 'cost', 'utm_campaign']]

# Ordenar por fecha de la más antigua a la más reciente
df_ads_combined = df_ads_combined.sort_values('date_group').reset_index(drop=True)

# ------------------------------------------
# 4. Guardar resultado en formato JSON
# ------------------------------------------
df_ads_combined.to_json("./ads.json")

print("✅ Archivo ads.json generado con éxito.")
print("\n--- Vista previa del DataFrame Final (Campañas) ---")
display(df_ads_combined.head(10))

In [ ]:
# ==========================================
# GRÁFICO 1 - VISITAS TOTALES
# ==========================================

import os
import matplotlib.pyplot as plt
import seaborn as sns

# Crear carpeta donde se guardarán los gráficos
os.makedirs("./charts", exist_ok=True)

# Estilo profesional
sns.set_theme(style="whitegrid")

# Agrupar visitas por fecha
visits_total = (
    df_conversion
    .groupby("date_group")["visits"]
    .sum()
    .reset_index()
)

# Crear gráfico
plt.figure(figsize=(15,6))

bars = plt.bar(
    visits_total["date_group"],
    visits_total["visits"],
    color="#4C72B0",
    edgecolor="black",
    linewidth=0.4,
    width=0.8
)

plt.title(
    "Total Visits Over Time",
    fontsize=16,
    fontweight="bold",
    pad=15
)

plt.xlabel("Date", fontsize=12)
plt.ylabel("Number of Visits", fontsize=12)

# Mostrar solo algunas fechas para evitar saturación
step = max(1, len(visits_total)//10)

plt.xticks(
    visits_total["date_group"][::step],
    visits_total["date_group"]
        .dt.strftime("%Y-%m-%d")[::step],
    rotation=45,
    ha="right"
)

plt.grid(axis="y", linestyle="--", alpha=0.35)
plt.grid(axis="x", visible=False)

sns.despine()

plt.tight_layout()

plt.savefig(
    "./charts/total_visits.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ==========================================
# GRÁFICO 2 - VISITAS POR PLATAFORMA
# (Barras apiladas)
# ==========================================

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# Agrupar visitas por fecha y plataforma
visits_platform = (
    df_conversion
    .groupby(["date_group", "platform"])["visits"]
    .sum()
    .unstack(fill_value=0)
)

# Mantener siempre el mismo orden de plataformas
platform_order = ["web", "android", "ios"]
visits_platform = visits_platform.reindex(columns=platform_order, fill_value=0)

# Colores consistentes
colors = {
    "web": "#4C72B0",
    "android": "#55A868",
    "ios": "#DD8452"
}

# Crear gráfico
fig, ax = plt.subplots(figsize=(15, 6))

visits_platform.plot(
    kind="bar",
    stacked=True,
    ax=ax,
    color=[colors[p] for p in platform_order],
    edgecolor="white",
    linewidth=0.3,
    width=0.8
)

ax.set_title(
    "Total Visits by Platform",
    fontsize=16,
    fontweight="bold",
    pad=15
)

ax.set_xlabel("Date", fontsize=12)
ax.set_ylabel("Number of Visits", fontsize=12)

# Mostrar pocas fechas para evitar saturación
step = max(1, len(visits_platform) // 10)

positions = list(range(0, len(visits_platform), step))

ax.set_xticks(positions)
ax.set_xticklabels(
    visits_platform.index.strftime("%Y-%m-%d")[::step],
    rotation=45,
    ha="right"
)

ax.legend(
    title="Platform",
    frameon=True,
    loc="upper left"
)

ax.grid(axis="y", linestyle="--", alpha=0.35)
ax.grid(axis="x", visible=False)

sns.despine()

plt.tight_layout()

plt.savefig(
    "./charts/visits_by_platform.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ==========================================
# GRÁFICO 3 - REGISTROS TOTALES
# ==========================================

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# Agrupar registros por fecha
registrations_total = (
    df_conversion
    .groupby("date_group")["registrations"]
    .sum()
    .reset_index()
)

plt.figure(figsize=(15,6))

plt.bar(
    registrations_total["date_group"],
    registrations_total["registrations"],
    color="#55A868",
    edgecolor="black",
    linewidth=0.4,
    width=0.8
)

plt.title(
    "Total Registrations Over Time",
    fontsize=16,
    fontweight="bold",
    pad=15
)

plt.xlabel("Date", fontsize=12)
plt.ylabel("Number of Registrations", fontsize=12)

step = max(1, len(registrations_total)//10)

plt.xticks(
    registrations_total["date_group"][::step],
    registrations_total["date_group"].dt.strftime("%Y-%m-%d")[::step],
    rotation=45,
    ha="right"
)

plt.grid(axis="y", linestyle="--", alpha=0.35)
plt.grid(axis="x", visible=False)

sns.despine()

plt.tight_layout()

plt.savefig(
    "./charts/total_registrations.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ==========================================
# GRÁFICO 4 - REGISTROS POR PLATAFORMA
# (Barras apiladas)
# ==========================================

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# Agrupar registros por fecha y plataforma
registrations_platform = (
    df_conversion
    .groupby(["date_group", "platform"])["registrations"]
    .sum()
    .unstack(fill_value=0)
)

platform_order = ["web", "android", "ios"]

registrations_platform = registrations_platform.reindex(
    columns=platform_order,
    fill_value=0
)

colors = {
    "web": "#4C72B0",
    "android": "#55A868",
    "ios": "#DD8452"
}

fig, ax = plt.subplots(figsize=(15,6))

registrations_platform.plot(
    kind="bar",
    stacked=True,
    ax=ax,
    color=[colors[p] for p in platform_order],
    edgecolor="white",
    linewidth=0.3,
    width=0.8
)

ax.set_title(
    "Registrations by Platform",
    fontsize=16,
    fontweight="bold",
    pad=15
)

ax.set_xlabel("Date", fontsize=12)
ax.set_ylabel("Number of Registrations", fontsize=12)

step = max(1, len(registrations_platform)//10)

positions = list(range(0, len(registrations_platform), step))

ax.set_xticks(positions)

ax.set_xticklabels(
    registrations_platform.index.strftime("%Y-%m-%d")[::step],
    rotation=45,
    ha="right"
)

ax.legend(
    title="Platform",
    frameon=True,
    loc="upper left"
)

ax.grid(axis="y", linestyle="--", alpha=0.35)
ax.grid(axis="x", visible=False)

sns.despine()

plt.tight_layout()

plt.savefig(
    "./charts/registrations_by_platform.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ==========================================
# GRÁFICO 5 - CONVERSIÓN POR PLATAFORMA
# ==========================================

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

plt.figure(figsize=(15,6))

sns.lineplot(
    data=df_conversion,
    x="date_group",
    y="conversion",
    hue="platform",
    palette={
        "web": "#4C72B0",
        "android": "#55A868",
        "ios": "#DD8452"
    },
    marker="o",
    linewidth=2.5,
    markersize=6
)

plt.title(
    "Conversion Rate by Platform",
    fontsize=16,
    fontweight="bold"
)

plt.xlabel("Date")
plt.ylabel("Conversion (%)")

step = max(1, len(df_conversion["date_group"].unique()) // 10)

dates = sorted(df_conversion["date_group"].unique())

plt.xticks(
    dates[::step],
    [d.strftime("%Y-%m-%d") for d in dates[::step]],
    rotation=45,
    ha="right"
)

plt.grid(axis="y", linestyle="--", alpha=0.35)

plt.legend(
    title="Platform",
    frameon=True
)

plt.tight_layout()

plt.savefig(
    "./charts/conversion_by_platform.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ==========================================
# GRÁFICO 6 - CONVERSIÓN PROMEDIO
# ==========================================

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

avg_conversion = (
    df_conversion
    .groupby("date_group")["conversion"]
    .mean()
    .reset_index()
)

plt.figure(figsize=(15,6))

plt.plot(
    avg_conversion["date_group"],
    avg_conversion["conversion"],
    color="#C44E52",
    linewidth=2.8,
    marker="o",
    markersize=6
)

# Área sombreada
plt.fill_between(
    avg_conversion["date_group"],
    avg_conversion["conversion"],
    alpha=0.20,
    color="#C44E52"
)

plt.title(
    "Average Conversion Rate Over Time",
    fontsize=16,
    fontweight="bold"
)

plt.xlabel("Date")
plt.ylabel("Average Conversion (%)")

step = max(1, len(avg_conversion)//10)

plt.xticks(
    avg_conversion["date_group"][::step],
    avg_conversion["date_group"].dt.strftime("%Y-%m-%d")[::step],
    rotation=45,
    ha="right"
)

plt.grid(axis="y", linestyle="--", alpha=0.35)

sns.despine()

plt.tight_layout()

plt.savefig(
    "./charts/average_conversion.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ==========================================
# GRÁFICO 7 - COSTOS DE CAMPAÑAS PUBLICITARIAS
# ==========================================

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

plt.figure(figsize=(15, 6))

plt.plot(
    df_ads_combined["date_group"],
    df_ads_combined["cost"],
    color="#7B68EE",
    linewidth=2.8,
    marker="o",
    markersize=5,
)

plt.fill_between(
    df_ads_combined["date_group"],
    df_ads_combined["cost"],
    color="#7B68EE",
    alpha=0.15,
)

plt.title("Advertising Campaign Costs", fontsize=16, fontweight="bold")

plt.xlabel("Date")
plt.ylabel("Cost")

step = max(1, len(df_ads_combined) // 10)

plt.xticks(
    df_ads_combined["date_group"][::step],
    df_ads_combined["date_group"].dt.strftime("%Y-%m-%d")[::step],
    rotation=45,
    ha="right",
)

plt.grid(axis="y", linestyle="--", alpha=0.35)

sns.despine()

plt.tight_layout()

plt.savefig(
    "./charts/advertising_costs.png", dpi=300, bbox_inches="tight"
)

plt.show()

In [ ]:
# ==========================================
# GRÁFICO 8 - VISITAS DURANTE CAMPAÑAS
# ==========================================

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

fig, ax = plt.subplots(figsize=(15, 6))

campaigns = df_ads_combined["utm_campaign"].unique()

palette = sns.color_palette("Pastel2", len(campaigns))

for campaign, color in zip(campaigns, palette):
  campaign_data = df_ads_combined[
      df_ads_combined["utm_campaign"] == campaign
  ]

  ax.axvspan(
      campaign_data["date_group"].min(),
      campaign_data["date_group"].max(),
      color=color,
      alpha=0.18,
      label=campaign,
  )

ax.plot(
    df_ads_combined["date_group"],
    df_ads_combined[
        "visits"
    ],  # Si aquí te da otro error, asegúrate de que sea la columna de visitas unida
    color="#1F77B4",
    linewidth=2.8,
    marker="o",
    markersize=4,
)

ax.set_title("Visits During Active Campaigns", fontsize=16, fontweight="bold")

ax.set_xlabel("Date")
ax.set_ylabel("Visits")

step = max(1, len(df_ads_combined) // 10)

ax.set_xticks(df_ads_combined["date_group"][::step])

ax.set_xticklabels(
    df_ads_combined["date_group"].dt.strftime("%Y-%m-%d")[::step],
    rotation=45,
    ha="right",
)

ax.grid(axis="y", linestyle="--", alpha=0.35)

handles, labels = ax.get_legend_handles_labels()

by_label = dict(zip(labels, handles))

ax.legend(
    by_label.values(),
    by_label.keys(),
    title="Campaign",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
)

plt.tight_layout()

plt.savefig("./charts/visits_during_campaigns.png", dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
# ==========================================
# GRÁFICO 9 - REGISTROS DURANTE CAMPAÑAS
# ==========================================

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

fig, ax = plt.subplots(figsize=(15, 6))

campaigns = df_ads_combined["utm_campaign"].unique()

palette = sns.color_palette("Pastel2", len(campaigns))

for campaign, color in zip(campaigns, palette):
  campaign_data = df_ads_combined[
      df_ads_combined["utm_campaign"] == campaign
  ]

  ax.axvspan(
      campaign_data["date_group"].min(),
      campaign_data["date_group"].max(),
      color=color,
      alpha=0.18,
      label=campaign,
  )

ax.plot(
    df_ads_combined["date_group"],
    df_ads_combined["registrations"],
    color="#2CA02C",
    linewidth=2.8,
    marker="o",
    markersize=4,
)

ax.set_title("Registrations During Active Campaigns", fontsize=16, fontweight="bold")

ax.set_xlabel("Date")
ax.set_ylabel("Registrations")

step = max(1, len(df_ads_combined) // 10)

ax.set_xticks(df_ads_combined["date_group"][::step])

ax.set_xticklabels(
    df_ads_combined["date_group"].dt.strftime("%Y-%m-%d")[::step],
    rotation=45,
    ha="right",
)

ax.grid(axis="y", linestyle="--", alpha=0.35)

handles, labels = ax.get_legend_handles_labels()

by_label = dict(zip(labels, handles))

ax.legend(
    by_label.values(),
    by_label.keys(),
    title="Campaign",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
)

plt.tight_layout()

plt.savefig(
    "./charts/registrations_during_campaigns.png", dpi=300, bbox_inches="tight"
)

plt.show()

# Presentación y conclusiones

## Objetivo

Analizar el comportamiento de las visitas, los registros, la conversión y los costos publicitarios para evaluar el desempeño de las campañas de marketing e identificar oportunidades de mejora.

---

## Hallazgo 1: Comportamiento de las visitas

**Gráfico:** `total_visits.png`

Las visitas presentan variaciones a lo largo del período analizado. Durante las campañas activas se observa un mayor volumen de tráfico, lo que evidencia el impacto de la publicidad en la adquisición de usuarios.

---

## Hallazgo 2: Registros y conversión

**Gráficos:** `total_registrations.png` y `conversion_by_platform.png`

Los registros siguen una tendencia similar a las visitas. La tasa de conversión presenta diferencias entre plataformas, lo que permite identificar oportunidades para optimizar el rendimiento de cada canal.

---

## Hallazgo 3: Costos publicitarios

**Gráfico:** `advertising_costs.png`

La inversión publicitaria varía durante el período analizado. Comparar estos costos con las visitas y los registros permite evaluar la eficiencia de las campañas.

---

## Hallazgo 4: Actividad durante las campañas

**Gráfico:** `visits_during_campaigns.png`

Las campañas activas coinciden con cambios en el comportamiento de las visitas, permitiendo analizar el impacto de cada período publicitario sobre el tráfico del sitio.

---

# Recomendaciones

- Mantener las campañas con mejor rendimiento.
- Optimizar las plataformas con menor tasa de conversión.
- Monitorear continuamente la relación entre visitas, registros y costos.
- Revisar el proceso de registro cuando disminuya la conversión.

---

# Conclusión

El análisis permitió evaluar el comportamiento de las campañas publicitarias mediante indicadores de visitas, registros, conversión y costos. Los resultados facilitan la toma de decisiones para optimizar la inversión y mejorar el rendimiento de futuras campañas.

In [ ]:
from IPython.display import Image, display

display(Image("./charts/total_visits.png"))
display(Image("./charts/total_registrations.png"))
display(Image("./charts/conversion_by_platform.png"))
display(Image("./charts/advertising_costs.png"))
display(Image("./charts/visits_during_campaigns.png"))